In [2]:
!pip install sentence-transformers


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import warnings
warnings.filterwarnings('ignore')

In [29]:
# pip install sentence-transformers scikit-learn numpy

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from kagglehub import KaggleDatasetAdapter, dataset_load

In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [4]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
val = pd.read_csv("val.csv")

train_x = train["text"].tolist()
train_y = train["labels"].values
test_x = test["text"].tolist()
test_y = test["labels"].values
val_x = val["text"].tolist()
val_y = val["labels"].values

In [5]:
train_x_embed=model.encode(train_x)
train_x_embed

array([[-0.04689604, -0.03568028,  0.08653586, ...,  0.02874709,
        -0.06657217,  0.0087614 ],
       [-0.00517401, -0.09443928, -0.06302016, ..., -0.02919849,
         0.07347699, -0.03061491],
       [-0.05326547,  0.07199333,  0.01703562, ..., -0.0026617 ,
         0.05330457, -0.01498123],
       ...,
       [-0.14286104, -0.00128124, -0.01009086, ...,  0.0248803 ,
        -0.11043192,  0.00938006],
       [ 0.03974754, -0.03598805,  0.00828503, ...,  0.04704768,
        -0.07953601, -0.03260503],
       [ 0.04467845, -0.09685981, -0.00914504, ...,  0.0001612 ,
        -0.03065691, -0.00630665]], shape=(13939, 384), dtype=float32)

In [11]:
pca = PCA(n_components=384)
train_x_reduced = pca.fit_transform(train_x_embed)
train_x_reduced

array([[-3.5018340e-01, -2.8866537e-02, -1.2726659e-01, ...,
        -8.7234291e-09,  2.3391667e-13, -4.6985913e-16],
       [ 2.5060293e-01,  1.2922834e-01, -3.1366004e-03, ...,
         2.8150033e-09, -7.5515657e-14, -1.0775857e-17],
       [-3.4000400e-01,  6.3985422e-02,  2.1862608e-01, ...,
        -2.5462619e-09,  6.8296009e-14,  4.0294363e-16],
       ...,
       [-2.9617509e-01,  1.6801585e-01,  4.0681735e-02, ...,
         1.8242083e-09, -4.8924460e-14, -1.2985554e-16],
       [-3.0740482e-01,  9.4805002e-02, -1.7821804e-01, ...,
         5.4817271e-09, -1.4715914e-13,  3.1445108e-16],
       [ 1.6985610e-01,  2.3124321e-01, -1.4059770e-01, ...,
        -6.9389827e-10,  1.8658089e-14, -2.0679940e-16]],
      shape=(13939, 384), dtype=float32)

In [14]:
clf = LogisticRegression(max_iter=1000, C=0.1, multi_class='auto')
clf.fit(train_x_reduced, train_y)

C:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.1
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'auto'


In [16]:
from sklearn.metrics import classification_report, accuracy_score

val_x_embed=model.encode(val_x)
y_val_pred = clf.predict(val_x_embed)

print("Validation Accuracy:", accuracy_score(val_y, y_val_pred))
print(classification_report(val_y, y_val_pred))

Validation Accuracy: 0.2525832376578645
              precision    recall  f1-score   support

           0       0.43      0.13      0.20       436
           1       0.25      0.28      0.26       435
           2       0.24      0.50      0.32       436
           3       0.22      0.10      0.14       435

    accuracy                           0.25      1742
   macro avg       0.28      0.25      0.23      1742
weighted avg       0.28      0.25      0.23      1742



In [21]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('clf', LogisticRegression(max_iter=1000, multi_class='auto'))
])

param_grid = {
    'pca__n_components': [100, 150, 200, 250, 300, 350, 400],
    'clf__C': [0.01, 0.1, 1]
}

grid = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy')
grid.fit(train_x_embed, train_y)

print("Best params:", grid.best_params_)
print("Validation score:", grid.best_score_)

Best params: {'clf__C': 0.01, 'pca__n_components': 350}
Validation score: 0.6847697555002644


In [22]:
test_x_embed=model.encode(test_x)
y_pred=grid.predict(test_x_embed)

In [23]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Accuracy:", accuracy_score(test_y, y_pred))
print("\nClassification Report:\n", classification_report(test_y, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(test_y, y_pred))

Accuracy: 0.6833046471600689

Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.82      0.79       435
           1       0.57      0.57      0.57       436
           2       0.75      0.73      0.74       436
           3       0.64      0.61      0.62       436

    accuracy                           0.68      1743
   macro avg       0.68      0.68      0.68      1743
weighted avg       0.68      0.68      0.68      1743


Confusion Matrix:
 [[356  25  38  16]
 [ 36 250  38 112]
 [ 49  43 320  24]
 [ 21 119  31 265]]
